# 🦿 Jev API + Gymnasium BipedalWalker — v3 Stable Hybrid

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/simple-jev/blob/main/notebooks/Jev_API_BipedalWalker_Controller_Colab.ipynb)

This version fixes the remaining forward-speed / forward-lean runaway seen in v2.

The v2 run accelerated from about 0.48 to 0.66 while hull angle deteriorated from about -0.075 to -0.467. By the time stabilize was selected, recovery was too late.

v3 keeps the 50 Hz low-level motor loop, asks Jev every 5 frames, predicts hull angle 100 ms ahead, progressively limits speed, and blends toward an upright two-leg recovery controller before the visible fall.

Create the API key at https://typesafe.ai and store it in Colab Secrets as TYPESAFE_API_KEY.

In [ ]:
#@title 1. Install dependencies
!apt-get update -qq
!apt-get install -y -qq swig > /dev/null
!pip -q install "gymnasium[box2d]" imageio imageio-ffmpeg typesafe-sdk pillow

In [ ]:
#@title 2. Imports, API key, and configuration
import os, time, json, getpass
from collections import deque
from pathlib import Path

import numpy as np
import gymnasium as gym
import imageio.v2 as imageio
import matplotlib.pyplot as plt

from PIL import Image as PILImage
from IPython.display import Video, Image, display
from gymnasium.envs.box2d.bipedal_walker import BipedalWalkerHeuristics
from typesafe_sdk import Choice, TypeSafeClient

def load_typesafe_key():
    key = None
    try:
        from google.colab import userdata
        key = userdata.get("TYPESAFE_API_KEY")
    except Exception:
        pass
    if not key:
        key = os.environ.get("TYPESAFE_API_KEY")
    if not key:
        key = getpass.getpass("TypeSafe API key from https://typesafe.ai: ").strip()
    if not key:
        raise RuntimeError("No TYPESAFE_API_KEY supplied.")
    return key

os.environ["TYPESAFE_API_KEY"] = load_typesafe_key()
os.environ.setdefault("TYPESAFE_DEFAULT_MODEL", "jev-latest")
client = TypeSafeClient()

SEED = 0
MAX_STEPS = 1600
DECISION_EVERY = 5
MIN_CONFIDENCE = 0.12

TARGET_SPEED = 0.29
SPEED_SOFT = 0.36
SPEED_HARD = 0.48
MOTOR_LIMIT = 0.95
MAX_ACTION_DELTA = 0.35

VIDEO_EVERY = 4
GIF_EVERY = 8
VIDEO_PATH = "/content/jev_bipedalwalker_v3.mp4"
GIF_PATH = "/content/jev_bipedalwalker_v3.gif"
LOG_PATH = "/content/jev_bipedalwalker_v3_decisions.json"

print("✓ TypeSafe key loaded")
print("✓ model:", os.environ["TYPESAFE_DEFAULT_MODEL"])
print("✓ Jev interval:", DECISION_EVERY, "frames; motor loop: 50 Hz")

In [ ]:
#@title 3. Jev supervisor + predictive balance controller
MODE_CRITERIA = {
    "cruise": (
        "Normal walking. Choose only when predicted hull angle is small, speed "
        "is near target, and no forward-fall trend is developing."
    ),
    "cautious": (
        "Reduce gait intensity before instability develops. Prefer when speed "
        "is above target, vertical motion grows, or the hull begins to lean."
    ),
    "balance": (
        "Blend strongly toward upright two-leg balance. Prefer when predicted "
        "angle or angular velocity indicates increasing instability."
    ),
    "recover": (
        "Prioritize staying upright over forward progress. Prefer when a fall "
        "is already developing or predicted soon."
    ),
}

def jev_choose_mode(state):
    t0 = time.perf_counter()
    response = client.system_one(
        state=state,
        questions={
            "control_mode": Choice(
                instructions=(
                    "Choose one supervisory mode for Gymnasium BipedalWalker-v3. "
                    "React early to predicted angle, angular velocity, excessive "
                    "forward speed, and recent trend. Priorities: avoid hull-ground "
                    "contact, stay upright, then walk right smoothly."
                ),
                criteria=MODE_CRITERIA,
            )
        },
    )
    latency_ms = (time.perf_counter() - t0) * 1000.0
    a = response.answers["control_mode"]
    return {
        "mode": a.choice,
        "confidence": float(a.confidence),
        "probabilities": dict(a.probabilities),
        "latency_ms": latency_ms,
        "model": response.model,
    }

def risk_metrics(s):
    angle = float(s[0])
    omega = float(s[1])
    vx = float(s[2])
    vy = float(s[3])

    # obs[1] = 2 * angular_velocity / 50, so 100 ms projection adds 2.5*obs[1].
    pred = angle + 2.5 * omega

    forward = float(np.clip((-pred - 0.06) / 0.24, 0.0, 1.0))
    backward = float(np.clip((pred - 0.10) / 0.24, 0.0, 1.0))
    tilt = max(forward, backward)
    speed = float(np.clip((vx - TARGET_SPEED) / (SPEED_HARD - TARGET_SPEED), 0.0, 1.0))
    vertical = float(np.clip((abs(vy) - 0.10) / 0.30, 0.0, 1.0))
    risk = float(max(tilt, 0.75 * speed, 0.55 * vertical))

    return {
        "angle": angle, "omega": omega, "vx": vx, "vy": vy,
        "pred": float(pred), "tilt_risk": tilt,
        "speed_risk": speed, "vertical_risk": vertical, "risk": risk,
    }

def upright_action(s):
    # Compact two-leg PD stance used only for balancing/recovery.
    hip_target = 0.0
    knee_target = 0.10
    hull = 1.10 * float(s[0]) + 1.90 * float(s[1])

    hip0 = 1.05 * (hip_target - float(s[4])) - 0.30 * float(s[5]) + hull
    knee0 = 4.0 * (knee_target - float(s[6])) - 0.30 * float(s[7]) - 11.0 * float(s[3])
    hip1 = 1.05 * (hip_target - float(s[9])) - 0.30 * float(s[10]) + hull
    knee1 = 4.0 * (knee_target - float(s[11])) - 0.30 * float(s[12]) - 11.0 * float(s[3])

    return np.clip(
        0.5 * np.array([hip0, knee0, hip1, knee1], dtype=np.float32),
        -MOTOR_LIMIT, MOTOR_LIMIT
    )

MODE_BLEND = {"cruise": 0.00, "cautious": 0.25, "balance": 0.55, "recover": 0.85}

def effective_mode(requested, m):
    if abs(m["angle"]) > 0.27 or abs(m["pred"]) > 0.33:
        return "recover", "emergency_tilt"
    if m["pred"] < -0.15:
        return "recover", "predicted_forward_fall"
    if m["angle"] < -0.12 and m["vx"] > SPEED_SOFT:
        return "recover", "speed_plus_forward_lean"
    if abs(m["pred"]) > 0.18:
        return "balance", "predicted_tilt"
    if m["vx"] > SPEED_HARD:
        return "balance", "hard_speed_limit"
    if m["vx"] > SPEED_SOFT and requested == "cruise":
        return "cautious", "soft_speed_limit"
    return requested, "jev"

def hybrid_action(reference_action, s, previous_action, requested_mode):
    m = risk_metrics(s)
    mode, reason = effective_mode(requested_mode, m)
    ref = np.asarray(reference_action, dtype=np.float32)
    stand = upright_action(s)

    alpha = MODE_BLEND.get(mode, 0.25)

    if m["pred"] < -0.08:
        alpha = max(alpha, float(np.clip((-m["pred"] - 0.08) / 0.20, 0.0, 0.90)))

    if m["vx"] > TARGET_SPEED:
        alpha = max(alpha, float(np.clip(
            (m["vx"] - TARGET_SPEED) / (SPEED_HARD - TARGET_SPEED), 0.0, 0.80
        )))

    action = (1.0 - alpha) * ref + alpha * stand

    prev = np.asarray(previous_action, dtype=np.float32)
    action = prev + np.clip(action - prev, -MAX_ACTION_DELTA, MAX_ACTION_DELTA)
    action = np.clip(action, -MOTOR_LIMIT, MOTOR_LIMIT).astype(np.float32)

    return action, mode, reason, alpha, m

def state_for_jev(obs, step, total_reward, recent_rewards, previous_mode):
    m = risk_metrics(obs)
    rr = list(recent_rewards)
    return {
        "environment": "Gymnasium BipedalWalker-v3",
        "goal": "walk right smoothly while preventing hull-ground contact",
        "step": int(step),
        "previous_mode": previous_mode,
        "total_reward": round(float(total_reward), 3),
        "recent_reward_mean": round(float(np.mean(rr)) if rr else 0.0, 4),
        "hull": {
            "angle": round(m["angle"], 4),
            "angular_velocity_scaled": round(m["omega"], 4),
            "predicted_angle_100ms": round(m["pred"], 4),
            "horizontal_speed_scaled": round(m["vx"], 4),
            "vertical_speed_scaled": round(m["vy"], 4),
        },
        "risk": {
            "overall": round(m["risk"], 3),
            "tilt": round(m["tilt_risk"], 3),
            "speed": round(m["speed_risk"], 3),
            "vertical": round(m["vertical_risk"], 3),
            "target_speed": TARGET_SPEED,
            "soft_speed_limit": SPEED_SOFT,
            "hard_speed_limit": SPEED_HARD,
        },
        "contacts": {"leg0": bool(obs[8] > 0.5), "leg1": bool(obs[13] > 0.5)},
    }

test = jev_choose_mode({
    "environment": "connectivity test",
    "hull": {"angle": 0.0, "predicted_angle_100ms": 0.0, "horizontal_speed_scaled": 0.1},
    "risk": {"overall": 0.0},
})
print("✓ Jev API:", test["mode"], "confidence", round(test["confidence"], 3))

In [ ]:
#@title 4. Run v3 stable hybrid controller
env = gym.make("BipedalWalker-v3", render_mode="rgb_array")
obs, info = env.reset(seed=SEED)

reference = BipedalWalkerHeuristics()
reference.SPEED = TARGET_SPEED

previous_action = np.zeros(4, dtype=np.float32)
requested_mode = "cruise"
recent_rewards = deque(maxlen=30)

total_reward = 0.0
decision_log, frame_log = [], []
reward_history, speed_history, angle_history = [], [], []
predicted_history, risk_history = [], []

api_failures = 0
low_confidence_fallbacks = 0
safety_overrides = 0
gif_frames = []

writer = imageio.get_writer(
    VIDEO_PATH,
    format="FFMPEG",
    mode="I",
    fps=max(1, 50 // VIDEO_EVERY),
    codec="libx264",
    pixelformat="yuv420p",
    macro_block_size=1,
)

try:
    for step in range(MAX_STEPS):
        if step % DECISION_EVERY == 0:
            state = state_for_jev(obs, step, total_reward, recent_rewards, requested_mode)
            try:
                d = jev_choose_mode(state)
                mode = d["mode"]
                if mode not in MODE_CRITERIA:
                    raise RuntimeError(f"Unknown Jev mode: {mode}")
                if d["confidence"] < MIN_CONFIDENCE:
                    mode = "cautious"
                    low_confidence_fallbacks += 1
                requested_mode = mode
                m = risk_metrics(obs)

                decision_log.append({
                    "step": step, "jev_mode": requested_mode,
                    "confidence": d["confidence"],
                    "probabilities": d["probabilities"],
                    "latency_ms": d["latency_ms"],
                    **m, "total_reward": float(total_reward),
                })

                if len(decision_log) <= 10 or len(decision_log) % 10 == 0:
                    print(
                        f"decision {len(decision_log):03d} | step {step:04d} | "
                        f"Jev={requested_mode:>8s} | conf {d['confidence']:.3f} | "
                        f"{d['latency_ms']:.0f} ms | vx {m['vx']:+.3f} | "
                        f"angle {m['angle']:+.3f} | pred {m['pred']:+.3f} | risk {m['risk']:.2f}"
                    )
            except Exception as e:
                api_failures += 1
                requested_mode = "cautious"
                print("Jev API error -> cautious:", repr(e))

        # Fresh Gymnasium reference action EVERY physics frame.
        reference_action = reference.step_heuristic(obs)

        action, applied_mode, reason, alpha, metrics = hybrid_action(
            reference_action, obs, previous_action, requested_mode
        )

        if reason != "jev":
            safety_overrides += 1

        frame_log.append({
            "step": step, "jev_mode": requested_mode,
            "applied_mode": applied_mode, "override_reason": reason,
            "blend_alpha": float(alpha), **metrics,
            "action": [float(x) for x in action],
        })

        obs, reward, terminated, truncated, info = env.step(action)
        previous_action = action.copy()
        total_reward += float(reward)
        recent_rewards.append(float(reward))

        ma = risk_metrics(obs)
        reward_history.append(total_reward)
        speed_history.append(ma["vx"])
        angle_history.append(ma["angle"])
        predicted_history.append(ma["pred"])
        risk_history.append(ma["risk"])

        if step % VIDEO_EVERY == 0:
            frame = env.render()
            writer.append_data(frame)

        if step % GIF_EVERY == 0:
            frame = env.render()
            gif_frames.append(np.asarray(PILImage.fromarray(frame).resize((450, 300))))

        if terminated or truncated:
            print(f"Episode ended at step {step + 1} | terminated={terminated} truncated={truncated}")
            break
finally:
    writer.close()
    env.close()

if gif_frames:
    imageio.mimsave(GIF_PATH, gif_frames, duration=GIF_EVERY / 50.0, loop=0)

Path(LOG_PATH).write_text(
    json.dumps({"decisions": decision_log, "frames": frame_log}, indent=2),
    encoding="utf-8",
)

print("\n=== RESULT ===")
print("steps:", len(reward_history))
print("total reward:", round(total_reward, 2))
print("Jev decisions:", len(decision_log))
print("low-confidence fallbacks:", low_confidence_fallbacks)
print("API failures:", api_failures)
print("safety-override frames:", safety_overrides)

if decision_log:
    lat = [d["latency_ms"] for d in decision_log]
    print("mean Jev latency (ms):", round(float(np.mean(lat)), 1))
    print("p95 Jev latency (ms):", round(float(np.percentile(lat, 95)), 1))

In [ ]:
#@title 5. Video, GIF fallback, and diagnostics
print("MP4:")
display(Video(VIDEO_PATH, embed=True, html_attributes="controls loop"))

print("GIF fallback:")
display(Image(filename=GIF_PATH))

plt.figure(figsize=(12, 4))
plt.plot(reward_history)
plt.xlabel("step"); plt.ylabel("cumulative reward")
plt.title("v3 cumulative reward"); plt.grid(True, alpha=0.25); plt.show()

plt.figure(figsize=(12, 4))
plt.plot(speed_history, label="horizontal speed")
plt.axhline(TARGET_SPEED, linestyle="--", linewidth=1, label="target")
plt.axhline(SPEED_SOFT, linestyle="--", linewidth=1, label="soft limit")
plt.axhline(SPEED_HARD, linestyle="--", linewidth=1, label="hard limit")
plt.xlabel("step"); plt.ylabel("scaled speed")
plt.title("Forward-speed governor"); plt.legend(); plt.grid(True, alpha=0.25); plt.show()

plt.figure(figsize=(12, 4))
plt.plot(angle_history, label="hull angle")
plt.plot(predicted_history, label="predicted angle +100 ms")
plt.axhline(-0.15, linestyle="--", linewidth=1)
plt.axhline(+0.18, linestyle="--", linewidth=1)
plt.xlabel("step"); plt.ylabel("radians")
plt.title("Predictive balance signal"); plt.legend(); plt.grid(True, alpha=0.25); plt.show()

plt.figure(figsize=(12, 4))
plt.plot(risk_history)
plt.xlabel("step"); plt.ylabel("risk 0..1"); plt.ylim(-0.02, 1.02)
plt.title("Predictive control risk"); plt.grid(True, alpha=0.25); plt.show()

print("MP4:", VIDEO_PATH)
print("GIF :", GIF_PATH)
print("log :", LOG_PATH)

## What changed from v2

v3 intervenes before the visible fall instead of waiting for a large hull angle.

Key changes:
1. Hull angle is projected 100 ms ahead from angular velocity.
2. Jev is queried every 5 frames instead of every 10.
3. Excess forward speed progressively blends the gait toward a stable two-leg controller.
4. Unsafe Jev decisions can be upgraded immediately by a 50 Hz safety governor.
5. Low Jev confidence now falls back to cautious, not aggressive cruise.
6. Gymnasium's own BipedalWalkerHeuristics is used directly.
7. A GIF fallback is generated if the browser cannot decode the MP4.

If it is still too fast, try TARGET_SPEED=0.26, SPEED_SOFT=0.32, SPEED_HARD=0.42.